# S2-PR-07 — Frozen T1 Evaluation Harness

This notebook is a public-safe walkthrough of the evaluator contract and validation evidence. It does not train a model and does not read TEST.


## 1. Goal
Create one deterministic T1 measuring stick reused by baselines, centralized models, and federated models.


In [ ]:
import json
from pathlib import Path

CONFIG = Path('config/evaluation/t1_evaluator.v1.json')
FIXTURE = Path('fixtures/evaluation/t1_hand_worked.v1.json')
SPEC = Path('docs/evidence/s2-pr-07/t1_metric_spec.v1.json')
RECORDS = Path('docs/evidence/s2-pr-07/t1_hand_worked_metric_records.v1.json')
VALIDATION = Path('docs/evidence/s2-pr-07/t1_evaluator_validation.v1.json')
DECISION = Path('docs/decisions/t1-evaluator-edge-semantics.md')

config = json.loads(CONFIG.read_text(encoding='utf-8'))
fixture = json.loads(FIXTURE.read_text(encoding='utf-8'))
spec = json.loads(SPEC.read_text(encoding='utf-8'))
records = json.loads(RECORDS.read_text(encoding='utf-8'))
validation = json.loads(VALIDATION.read_text(encoding='utf-8'))
print(validation['status'], validation['freeze_status'])


## 2. Frozen upstream protocol
- 588 dense categories.
- Macro is headline; micro is always shown.
- T1 regime-comparison headline is MRR@20 on category-change decisions.
- Overall metrics are diagnostic only.


## 3. T1 label semantics
`label_value` is already the dense TRAIN category code `0..587`; it is not looked up as a raw category ID.


## 4. Target-rank definition
Evaluation is exhaustive over the full vocabulary. Exact ties are ordered by ascending dense category code. This tie rule is approved (edge rule 3), not provisional.


## 5. Metrics
`Accuracy@1 = 1[r=1]`. `MRR@20 = 1/r` for `r<=20`, otherwise `0`.


## 6. Macro vs micro
Macro first averages decisions inside each client, then gives each supported client one vote. Micro gives each decision one vote.


In [ ]:
fixture['expected']


## 7. Category-change headline vs overall diagnostic


In [ ]:
spec['metric_ids']

{
    'freeze_status': spec['freeze_status'],
    'adr_001_history_buckets': spec['adr_001_history_buckets'],
    'approved_edge_history_bucket': spec['approved_edge_history_bucket'],
}


## 8. History buckets
History stratification uses externally supplied TRAIN event counts only. The evaluator never substitutes T1 example counts.

`ADR-001` freezes four buckets: `10_19`, `20_49`, `50_99`, `100_plus`. `BELOW_10_RETAINED_C1` is not an `ADR-001` bucket; it is the approved S2-PR-07 edge extension (rule 12) that discloses retained `C1` clients whose supplied TRAIN-event count is below 10 instead of dropping them or folding them into `10_19`.

The spec separates the two so provenance is never inferred from the combined list.


## 9. Edge semantics and approval state


In [ ]:
print(DECISION.read_text(encoding='utf-8'))


## 10. Real VALIDATION TaskExample preflight
Only public-safe aggregates are displayed. The notebook itself does not read private parquet.


In [ ]:
validation['real_validation_preflight']


## 11. Hand-worked metric records


In [ ]:
records


## 12. Validation evidence


In [ ]:
validation


## 13. What this does not do
No model training, no TEST evaluation, no R1/R2A result, no QR, and no extra metric freeze.


## 14. Downstream consumers
#32 uses this evaluator for T1 baselines; #33 uses it for the centralized pilot; #54 uses the frozen evaluator for final R1.
